In [23]:
!pip install rdkit
!pip install catboost

In [24]:
import kagglehub
kagglehub.login()

In [25]:
import numpy as np
import pandas as pd

seed = 42

#Базовые файлики
path = kagglehub.competition_download('molecular-activity-prediction-hack')
print("Path to competition files:", path)

train = pd.read_csv(path+'/train_v2.csv')
test = pd.read_csv(path+'/test.csv')
train

Path to competition files: /root/.cache/kagglehub/competitions/molecular-activity-prediction-hack


,Title,CC50_uM,IC50_uM_VACV,SI VACV,SMILES,strain,Culture
0,PAV-825,15.0,2.18,6.881,CCC(CC)C1=CC2=C(C=C1)N=C1C(C(C)(C)C)=CC(N(CC)C...,"Vaccinia virus Western Reserve, HeLa cells",HeLa cells
1,11c,26.2,NaN,0.000,C[C@@]12CC[C@H](C2(C)C)C[C@H]1OC(=O)c3ccc(Br)cc3,Vaccinia virus (strain Copenhagen). culture Ve...,Vero
2,10a,217.2,NaN,0.000,C1C[C@@H](C2)C(C)(C)[C@@H]([C@]12C)NC(=O)c3ccccc3,Vaccinia virus (strain Copenhagen). culture Ve...,Vero
3,sid103692656,100.0,100.00,1.000,O=C(c1ccccc1Br)n1ccnc1-c1ccc(Cl)cc1,"Vaccinia virus, HFF cells",Human foreskin fibroblast
4,sid160666886,45.0,NaN,0.000,CC(C(=O)NNC(=O)NC1CCCCC1)c1cccc(Cc2ccccc2)c1,"Vaccinia virus, HELA cells",HELA cells
...,...,...,...,...,...,...,...
315,7j-as-526,104.5,6.80,15.368,CC1(C)[C@@H]2CC[C@@]1(C)[C@H](NC(=O)C1=CN=CC=C...,Vaccinia virus (strain Copenhagen). culture Ve...,Vero
316,11,258.0,38.70,6.667,CC1(C)[C@@H]2CC[C@@]1(C)/C(=N/NC(=O)C1=CC=CO1)C2,Vaccinia virus (strain Copenhagen). culture Vero,Vero
317,sid134450510,303.0,NaN,0.000,Cc1cc(C)cc(C(=O)c2cc(Cl)ccc2OCCn2c(C)cc(=O)[nH...,Vaccinia virus,MT-4
318,sid312354110,100.0,23.00,4.348,Cc1nc2cc(C(F)(F)F)ccc2n1C[C@@H]1CCCN2CCCC[C@H]12,"Vaccinia virus, Vero cells",Vero


In [26]:
from rdkit import Chem
from rdkit.Chem import Descriptors
train['MolWt'] = train['SMILES'].apply(lambda x: Descriptors.MolWt(Chem.MolFromSmiles(x)))
train['LogP'] = train['SMILES'].apply(lambda x: Descriptors.MolLogP(Chem.MolFromSmiles(x)))
train['NumAtoms'] = train['SMILES'].apply(lambda x: Chem.MolFromSmiles(x).GetNumAtoms())
train['RingCount'] = train['SMILES'].apply(lambda x: Descriptors.RingCount(Chem.MolFromSmiles(x)))
train['NumHDonors'] = train['SMILES'].apply(lambda x: Descriptors.NumHDonors(Chem.MolFromSmiles(x)))
train['NumHAcceptors'] = train['SMILES'].apply(lambda x: Descriptors.NumHAcceptors(Chem.MolFromSmiles(x)))
train['TPSA'] = train['SMILES'].apply(lambda x: Descriptors.TPSA(Chem.MolFromSmiles(x)))

test['MolWt'] = test['SMILES'].apply(lambda x: Descriptors.MolWt(Chem.MolFromSmiles(x)))
test['LogP'] = test['SMILES'].apply(lambda x: Descriptors.MolLogP(Chem.MolFromSmiles(x)))
test['NumAtoms'] = test['SMILES'].apply(lambda x: Chem.MolFromSmiles(x).GetNumAtoms())
test['RingCount'] = test['SMILES'].apply(lambda x: Descriptors.RingCount(Chem.MolFromSmiles(x)))
test['NumHDonors'] = test['SMILES'].apply(lambda x: Descriptors.NumHDonors(Chem.MolFromSmiles(x)))
test['NumHAcceptors'] = test['SMILES'].apply(lambda x: Descriptors.NumHAcceptors(Chem.MolFromSmiles(x)))
test['TPSA'] = test['SMILES'].apply(lambda x: Descriptors.TPSA(Chem.MolFromSmiles(x)))

In [27]:
from rdkit.Chem import rdFingerprintGenerator

fpsize = 1024
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=fpsize)
fp_columns = [f'M_{i}' for i in range(fpsize)]

train_fp = np.array([list(generator.GetFingerprint(Chem.MolFromSmiles(s))) for s in train['SMILES']])
test_fp = np.array([list(generator.GetFingerprint(Chem.MolFromSmiles(s))) for s in test['SMILES']])

train[fp_columns] = train_fp
test[fp_columns] = test_fp

/tmp/ipykernel_2102/60078016.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[fp_columns] = train_fp
/tmp/ipykernel_2102/60078016.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[fp_columns] = train_fp
/tmp/ipykernel_2102/60078016.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`

In [28]:
#Удаляю ненужные признаки
train.drop(['SMILES', 'Title', 'CC50_uM', 'IC50_uM_VACV'], axis=1, inplace=True)
test.drop(['SMILES', 'Title', 'CC50_uM', 'IC50_uM_VACV'], axis=1, inplace=True)

In [29]:
target = 'SI VACV'
cat_cols = ['strain', 'Culture']
num_cols = ['MolWt', 'LogP', 'NumAtoms', 'RingCount', 'NumHDonors', 'NumHAcceptors', 'TPSA']

In [30]:
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler()
        ), num_cols)
    ],
    verbose_feature_names_out=False
)

X_train = train.drop(columns=[target])
y_train = train[target]
X_test = test.copy()

train_features_prep = preprocessor.fit_transform(X_train)
test_features_prep = preprocessor.transform(X_test)

train_prep_df = pd.DataFrame(train_features_prep, columns=preprocessor.get_feature_names_out(), index=train.index)
test_prep_df = pd.DataFrame(test_features_prep, columns=preprocessor.get_feature_names_out(), index=test.index)

train_prep = pd.concat([train_prep_df, train[fp_columns]], axis=1)
test_prep = pd.concat([test_prep_df, test[fp_columns]], axis=1)

print("Размерность тренировочной выборки:", train_prep.shape)
print("Размерность тестовой выборки:", test_prep.shape)

Размерность тренировочной выборки: (320, 1090)
Размерность тестовой выборки: (597, 1090)


In [35]:
X = train_prep
y = train[target]

In [45]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(random_state=seed)
model.fit(X, y)
y_pred = model.predict(test_prep)

In [49]:
submission = pd.DataFrame({
    'index': test_prep.index,
    target: y_pred
})

submission.to_csv('submission.csv', index=False)